In [1]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import sqlite3

load_dotenv()
print("✅ Ready for human-in-the-loop workflows with LangGraph 1.1.9!")

✅ Ready for human-in-the-loop workflows with LangGraph 1.1.9!


### **Approval Workflow with interrupt() + Command(resume=...)**

In [2]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class ApprovalState(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

In [3]:
def plan_task(state: ApprovalState) -> dict:
    """Generate a plan and ask for approval."""
    plan = f"I will execute: {state['task']} using the following steps: 1. Setup, 2. Execute, 3. Verify"
    print(f"📋 Generated plan: {plan}")

    approval = interrupt({
        "question": "Do you approve this plan?",
        "plan": plan,
        "options": ["yes", "no", "modify"],
    })

    
    print(f"👤 Human responded: {approval}")
    return {"plan": plan, "approved": approval == "yes"}

def execute_task(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Task cancelled by user"}
    result = f"✅ Successfully executed: {state['task']}"
    print(result)
    return {"result": result}

In [4]:
# Build graph
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

# Build graph
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_task)
builder.add_node("execute", execute_task)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)

app = builder.compile(checkpointer=memory)
print("✅ Approval workflow (modern pattern) ready!")

✅ Approval workflow (modern pattern) ready!


In [5]:
config = {"configurable": {"thread_id": "task-001"}}

# STEP 1: Run until interrupt ---
print("=== Step 1: Starting task ===")

result = app.invoke(
    {"task": "Deploy database migration", "plan": "", "approved": False, "result": ""},
    config
)

# The result contains the interrupt info
print(f"\nGraph paused. Interrupt data: {result.get('__interrupt__', 'check state')}")

=== Step 1: Starting task ===
📋 Generated plan: I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify

Graph paused. Interrupt data: [Interrupt(value={'question': 'Do you approve this plan?', 'plan': 'I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify', 'options': ['yes', 'no', 'modify']}, id='55442937ffaefa479fd07f340b5289fe')]


In [6]:
# --- STEP 2: Inspect state while paused ---
state = app.get_state(config)
print(f"\nCurrent state: task='{state.values['task']}'")
print("Waiting for human approval...")


Current state: task='Deploy database migration'
Waiting for human approval...


In [7]:
# --- STEP 3: Resume with human decision using Command(resume=...) ---
print("\n=== Step 3: Resuming with approval ===")
final_result = app.invoke(Command(resume="yes"), config)
print(f"\nFinal result: {final_result['result']}")


=== Step 3: Resuming with approval ===
📋 Generated plan: I will execute: Deploy database migration using the following steps: 1. Setup, 2. Execute, 3. Verify
👤 Human responded: yes
✅ Successfully executed: Deploy database migration

Final result: ✅ Successfully executed: Deploy database migration


In [8]:
import sqlite3
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

In [9]:
class ApprovalState(TypedDict):
    task: str
    plan: str
    approved: bool
    result: str

In [10]:
def plan_task(state: ApprovalState) -> dict:
    plan = (
        f"Plan for '{state['task']}':\n"
        "  Step 1 — Validate inputs\n"
        "  Step 2 — Execute operation\n"
        "  Step 3 — Verify results"
    )
    print(f"\n📋 Generated plan:\n{plan}\n")
    approval = interrupt({
        "question": "Do you approve this plan?",
        "options": ["yes", "no", "modify"],
    })
    print(f"👤 Human responded: {approval}")
    return {"plan": plan, "approved": (approval == "yes")}

In [11]:
def execute_task(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Task cancelled by user."}
    return {"result": f"✅ Executed: {state['task']}"}

In [12]:
db = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ApprovalState)
builder.add_node("plan", plan_task)
builder.add_node("execute", execute_task)
builder.add_edge(START, "plan")
builder.add_edge("plan", "execute")
builder.add_edge("execute", END)
app = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "approval-live-01"}}

In [13]:
app.invoke(
    {"task": "Deploy database migration", "plan": "", "approved": False, "result": ""},
    config,
)


📋 Generated plan:
Plan for 'Deploy database migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results



{'task': 'Deploy database migration',
 'plan': '',
 'approved': False,
 'result': '',
 '__interrupt__': [Interrupt(value={'question': 'Do you approve this plan?', 'options': ['yes', 'no', 'modify']}, id='9ef341500c63ec280c75377a2d15886c')]}

In [14]:
human_answer = input("⌨️  Your decision (yes / no / modify): ").strip().lower()

In [15]:
final = app.invoke(Command(resume=human_answer), config)
print(f"\n🏁 Final result: {final['result']}")


📋 Generated plan:
Plan for 'Deploy database migration':
  Step 1 — Validate inputs
  Step 2 — Execute operation
  Step 3 — Verify results

👤 Human responded: yes

🏁 Final result: ✅ Executed: Deploy database migration


### **Static vs Dynamic Interrupts**



In [ ]:
# **Static Interrupts (compile-time)**
# Set breakpoints when compiling the graph — always interrupt at these nodes:

# app = graph.compile(
#     checkpointer=memory,
#     interrupt_before=["execute_payment"],   # Pause BEFORE this node runs
#     interrupt_after=["classify_intent"],    # Pause AFTER this node runs
# )

# **Dynamic Interrupts (runtime, recommended)**
# The interrupt() function inside nodes — conditional and context-aware:

# def execute_payment(state):
#     if state["amount"] > 1000:
#         approval = interrupt(f"Large payment of ${state['amount']} — approve?")
#         if approval != "yes":
#             return {"status": "declined"}
#     # Proceed with payment...

### **Static Interrupts: interrupt_before Example**

In [17]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import TypedDict

class PaymentState(TypedDict):
    amount: float
    merchant: str
    status: str

In [18]:
def validate_payment(state: PaymentState) -> dict:
    print(f"💳 Validating payment: ${state['amount']} to {state['merchant']}")
    return {"status": "validated"}


def process_payment(state: PaymentState) -> dict:
    print(f"✅ Processing payment: ${state['amount']}")
    return {"status": "processed"}

In [19]:
db2 = sqlite3.connect(":memory:", check_same_thread=False)
memory2 = SqliteSaver(db2)

builder2 = StateGraph(PaymentState)
builder2.add_node("validate", validate_payment)
builder2.add_node("process", process_payment)
builder2.add_edge(START, "validate")
builder2.add_edge("validate", "process")
builder2.add_edge("process", END)


In [20]:
app2 = builder2.compile(
    checkpointer=memory2,
    interrupt_before=["process"]  # ← static breakpoint
)

config2 = {"configurable": {"thread_id": "payment-001"}}

In [21]:
# Run to the interrupt
print("=== Running to interrupt ===")
app2.invoke({"amount": 5000.0, "merchant": "AWS", "status": "pending"}, config2)

print("\n⏸️  Paused before 'process' node. Human review required.")

=== Running to interrupt ===
💳 Validating payment: $5000.0 to AWS

⏸️  Paused before 'process' node. Human review required.


In [22]:
print("\n=== Resuming ===")
final = app2.invoke(None, config2)
print(f"Final status: {final['status']}")


=== Resuming ===
✅ Processing payment: $5000.0
Final status: processed


### **Streaming Modes — Complete Reference**

In [24]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage

def step1(state: MessagesState) -> dict:
    return {"messages": [{"role": "assistant", "content": "Step 1 complete"}]}

def step2(state: MessagesState) -> dict:
    return {"messages": [{"role": "assistant", "content": "Step 2 complete"}]}

stream_builder = StateGraph(MessagesState)
stream_builder.add_node("step1", step1)
stream_builder.add_node("step2", step2)
stream_builder.add_edge(START, "step1")
stream_builder.add_edge("step1", "step2")
stream_builder.add_edge("step2", END)
stream_app = stream_builder.compile()

print("\n=== stream_mode='updates' — only changes from each node ===")
for chunk in stream_app.stream(
    {"messages": [HumanMessage("Go")]},
    stream_mode="updates"
):
    print(f"Node update: {chunk}")


=== stream_mode='updates' — only changes from each node ===
Node update: {'step1': {'messages': [{'role': 'assistant', 'content': 'Step 1 complete'}]}}
Node update: {'step2': {'messages': [{'role': 'assistant', 'content': 'Step 2 complete'}]}}


In [26]:
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class ProcessState(TypedDict):
    items: list
    processed: list

def process_items(state: ProcessState) -> dict:
    """Node that emits custom progress events during processing."""
    writer = get_stream_writer() 


    processed = []
    for i, item in enumerate(state["items"]):
        result = f"processed_{item}"
        processed.append(result)

        # Emit custom progress event
        writer({"type": "progress", "item": item, "done": i + 1, "total": len(state["items"])})

    return {"processed": processed}

custom_builder = StateGraph(ProcessState)
custom_builder.add_node("process", process_items)
custom_builder.add_edge(START, "process")
custom_builder.add_edge("process", END)
custom_app = custom_builder.compile()


print("=== stream_mode='custom' — custom events from nodes ===")
for event in custom_app.stream(
    {"items": ["alpha", "beta", "gamma"], "processed": []},
    stream_mode="custom"
):
    print(f"Progress: {event['item']} ({event['done']}/{event['total']})")

=== stream_mode='custom' — custom events from nodes ===
Progress: alpha (1/3)
Progress: beta (2/3)
Progress: gamma (3/3)


In [27]:
# Combining multiple stream modes
print("=== Combining stream modes: ['updates', 'custom'] ===")
for chunk in custom_app.stream(
    {"items": ["x", "y", "z"], "processed": []},
    stream_mode=["updates", "custom"]  # ← list of modes
):
    # When combining modes, each chunk is a tuple: (mode_name, data)
    print(f"Chunk: {chunk}")

=== Combining stream modes: ['updates', 'custom'] ===
Chunk: ('custom', {'type': 'progress', 'item': 'x', 'done': 1, 'total': 3})
Chunk: ('custom', {'type': 'progress', 'item': 'y', 'done': 2, 'total': 3})
Chunk: ('custom', {'type': 'progress', 'item': 'z', 'done': 3, 'total': 3})
Chunk: ('updates', {'process': {'processed': ['processed_x', 'processed_y', 'processed_z']}})


In [28]:
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

def chat(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

chat_builder = StateGraph(MessagesState)
chat_builder.add_node("chat", chat)
chat_builder.add_edge(START, "chat")
chat_builder.add_edge("chat", END)
chat_app = chat_builder.compile()

print("=== stream_mode='messages' — LLM tokens as they arrive ===")
print("Response: ", end="", flush=True)
for chunk, metadata in chat_app.stream(
    {"messages": [HumanMessage("Count from 1 to 200, one number per word.")]},
    stream_mode="messages"
):
    if hasattr(chunk, "content") and chunk.content:
        print(chunk.content, end="", flush=True)
print()

=== stream_mode='messages' — LLM tokens as they arrive ===
Response: Sure! Here’s the count from 1 to 200, one number per word:

1 2 3 4 5 6 7 8 9 10  
11 12 13 14 15 16 17 18 19 20  
21 22 23 24 25 26 27 28 29 30  
31 32 33 34 35 36 37 38 39 40  
41 42 43 44 45 46 47 48 49 50  
51 52 53 54 55 56 57 58 59 60  
61 62 63 64 65 66 67 68 69 70  
71 72 73 74 75 76 77 78 79 80  
81 82 83 84 85 86 87 88 89 90  
91 92 93 94 95 96 97 98 99 100  
101 102 103 104 105 106 107 108 109 110  
111 112 113 114 115 116 117 118 119 120  
121 122 123 124 125 126 127 128 129 130  
131 132 133 134 135 136 137 138 139 140  
141 142 143 144 145 146 147 148 149 150  
151 152 153 154 155 156 157 158 159 160  
161 162 163 164 165 166 167 168 169 170  
171 172 173 174 175 176 177 178 179 180  
181 182 183 184 185 186 187 188 189 190  
191 192 193 194 195 196 197 198 199 200  

There you go!


### **Mini Project: Research Assistant with HITL + Streaming**

In [29]:
from langchain_core.tools import tool

class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    findings: list
    approved_sources: list

@tool
def search_papers(query: str) -> str:
    """Search for academic papers (simulated)."""
    papers = [
        "Paper 1: Introduction to Quantum Computing",
        "Paper 2: Quantum Algorithms Overview",
        "Paper 3: Practical Quantum Applications"
    ]
    return "\n".join(papers)

@tool
def summarize_paper(title: str) -> str:
    """Summarize a paper (simulated, expensive operation)."""
    return f"Summary of {title}: [Detailed summary would go here]"


def research_node(state: ResearchState) -> dict:
    """Conduct research with human oversight."""
    # Search for papers
    results = search_papers.invoke({"query": state["topic"]})
    
    # Ask for clarification if needed
    clarification = interrupt(
        f"Found these papers about {state['topic']}. \n{results}\n\n"
        "Should I continue with detailed analysis? (yes/no)"
    )
    
    if clarification == "yes":
        # Proceed with analysis
        findings = ["Finding 1", "Finding 2", "Finding 3"]
        return {"findings": findings}
    
    return {"findings": []}

# Build research assistant
research_builder = StateGraph(ResearchState)
research_builder.add_node("research", research_node)
research_builder.add_edge(START, "research")
research_builder.add_edge("research", END)

research_db = sqlite3.connect(":memory:", check_same_thread=False)
research_memory = SqliteSaver(research_db)
research_assistant = research_builder.compile(checkpointer=research_memory)

print("✅ Research Assistant ready!")

✅ Research Assistant ready!


In [30]:
from langgraph.types import Command

# 1. We must provide a thread_id because the checkpointer requires it to save the paused state
config = {"configurable": {"thread_id": "research-thread-001"}}

initial_state = {
    "topic": "Quantum Computing",
    "messages": [],
    "findings": [],
    "approved_sources": []
}

print("=" * 60)
print("1. STARTING THE GRAPH")
print("=" * 60)
# The graph will run until it hits the interrupt(), then pause and return nothing.
research_assistant.invoke(initial_state, config)

1. STARTING THE GRAPH


{'messages': [],
 'topic': 'Quantum Computing',
 'findings': [],
 'approved_sources': [],
 '__interrupt__': [Interrupt(value='Found these papers about Quantum Computing. \nPaper 1: Introduction to Quantum Computing\nPaper 2: Quantum Algorithms Overview\nPaper 3: Practical Quantum Applications\n\nShould I continue with detailed analysis? (yes/no)', id='8b1e4af5923ba6783ac982560eea3cd1')]}

In [31]:
# 2. Check the current state to see what the interrupt is asking
state = research_assistant.get_state(config)

# state.tasks holds the paused execution details. We extract the interrupt message here.
interrupt_message = state.tasks[0].interrupts[0].value
print(f"🤖 Assistant asks:\n{interrupt_message}")


print("\n" + "=" * 60)
print("2. RESUMING THE GRAPH")
print("=" * 60)


🤖 Assistant asks:
Found these papers about Quantum Computing. 
Paper 1: Introduction to Quantum Computing
Paper 2: Quantum Algorithms Overview
Paper 3: Practical Quantum Applications

Should I continue with detailed analysis? (yes/no)

2. RESUMING THE GRAPH


In [32]:
# 3. Resume the graph by passing the string "yes" back into the interrupt
print("🧑 Human says: yes\n")
final_state = research_assistant.invoke(Command(resume="yes"), config)

print("✅ Final Findings:")
print(final_state["findings"])

🧑 Human says: yes

✅ Final Findings:
['Finding 1', 'Finding 2', 'Finding 3']


In [33]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage
from langchain_openai import AzureChatOpenAI

llm_v2 = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

def chat_v2(state: MessagesState) -> dict:
    response = llm_v2.invoke(state["messages"])
    return {"messages": [response]}

v2_builder = StateGraph(MessagesState)
v2_builder.add_node("chat", chat_v2)
v2_builder.add_edge(START, "chat")
v2_builder.add_edge("chat", END)
v2_app = v2_builder.compile()

In [34]:
print("=== version='v2' streaming — typed StreamPart chunks ===")
for chunk in v2_app.stream(
    {"messages": [HumanMessage("Say hello")]},
    stream_mode="updates",
    version="v2"  # ← opt-in to type-safe streaming
):
    # chunk is now a StreamPart TypedDict:
    # {"type": "updates"|"values"|..., "ns": (), "data": ...}
    print(f"Type: {chunk['type']}, Namespace: {chunk['ns']}, Data keys: {list(chunk['data'].keys())}")

=== version='v2' streaming — typed StreamPart chunks ===
Type: updates, Namespace: (), Data keys: ['chat']


In [35]:
print("\n=== version='v2' invoke — returns GraphOutput dataclass ===")
result = v2_app.invoke(
    {"messages": [HumanMessage("Say hello")]},
    version="v2"
)


=== version='v2' invoke — returns GraphOutput dataclass ===


In [36]:
# result is now a GraphOutput, not a plain dict:
print(f"Type: {type(result)}")
print(f"State value: {result.value['messages'][-1].content}")
print(f"Interrupts: {result.interrupts}")  # Any interrupt objects if graph was paused

Type: <class 'langgraph.types.GraphOutput'>
State value: Hello there! How can I assist you today? 😊
Interrupts: ()
